In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import glob, os
import pandas as pd

import scipy.ndimage
import scienceplots
plt.style.use(['science', 'notebook', 'grid'])

In [ ]:
# function to do moving average
def moving_average(x, y, w):
    y = np.convolve(y, np.ones(w), 'valid') / w
    x = x[w-1:]
    return x, y

def crop(x, y, x_min=None, x_max=None):
    if x_min is not None:
        arg_min = x > x_min
        x = x[arg_min]
        y = y[arg_min]
    if x_max is not None:
        arg_max = x < x_max
        x = x[arg_max]
        y = y[arg_max]
    return x, y

In [ ]:
folder = "../../build/iv_pn_2d/**/"
files = glob.glob(
    os.path.join(folder, "self_consistent_PBMC_history.csv")
)
files.sort()
print(f"Found {len(files)} files.")
list_vapplied = []
for file in files:
    df = pd.read_csv(file)
    v = df["ramo_electrode_voltage_V"].values[0]
    list_vapplied.append(v)
argsort = np.argsort(list_vapplied)
list_vapplied = [list_vapplied[i] for i in argsort]
files = [files[i] for i in argsort]
print(f"Sorted files by applied voltage: {list_vapplied}, minmax: {min(list_vapplied)}, {max(list_vapplied)}")

In [ ]:
# fig2, axs2 = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
fig, axs = plt.subplots(3, 1, figsize=(10, 8), sharex=True)
Nsmooth = 100
nbfiles = len(files)
colors = plt.cm.jet(np.linspace(list_vapplied[0], list_vapplied[-1], nbfiles))
list_mean_final_current = []
for idx, file in enumerate(files[:]):
    try:
        # print(f"Reading {file}...")
        data = pd.read_csv(file)
    except Exception as e:
        print(f"Error reading {file}: {e}")
        continue

    time = data["time"].values * 1e12 # convert to ps
    nb_electrons = data["nb_electrons"].values
    nb_holes = data["nb_holes"].values
    ramo_total_current = data["ramo_current"].values
    ramo_electrode_voltage = data["ramo_electrode_voltage_V"].values
    reference_electrode_voltage = data["reference_electrode_voltage_V"].values
    quench_bias_voltage = data["quench_bias_voltage_V"].values
    quench_device_current = data["quench_device_current_A"].values
    quench_voltage_drop = data["quench_voltage_drop_V"].values
    v_applied = data["ramo_electrode_voltage_V"].values
    
    
    tmin = .0
    argtmin = time > tmin
    time = time[argtmin]
    nb_electrons = nb_electrons[argtmin]
    nb_holes = nb_holes[argtmin]
    ramo_total_current = ramo_total_current[argtmin]
    anode_voltage = anode_voltage[argtmin]
    cathode_voltage = cathode_voltage[argtmin]
    quench_bias_voltage = quench_bias_voltage[argtmin]
    quench_device_current = quench_device_current[argtmin]
    quench_voltage_drop = quench_voltage_drop[argtmin]
    mean_final_current = np.mean(ramo_total_current[-100:])
    list_mean_final_current.append(mean_final_current)

    axs[0].plot(time, ramo_total_current, label="Ramo current",c=colors[idx])
    axs[1].plot(time, anode_voltage, label="Anode voltage",c=colors[idx])
    # axs[1].plot(time, cathode_voltage, label="Cathode voltage")
    axs[2].plot(time, nb_holes + nb_electrons, label="Total carriers",c=colors[idx])


axs[-1].set_xlabel("Time (ps)")
axs[0].set_ylabel("Current (A)")
axs[1].set_ylabel("Voltage (V)")
axs[2].set_ylabel("Number of carriers")


plt.show()


In [ ]:
# plt.tight_layout()

fig2, axs2 = plt.subplots(1, 1, figsize=(6, 4), sharex=True)
axs2.plot(list_vapplied, np.array(list_mean_final_current), marker='.')
axs2.set_xlabel("Applied voltage (V)")
axs2.set_ylabel("Mean final current (A)")
axs2.set_xscale('linear')
# axs2.set_yscale('log')
plt.show()